In [1]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter

pd.set_option('display.max_colwidth', 120)

In [3]:
DATASET_PATH = Path('data')
PROCESSED_PATH = DATASET_PATH / 'processed'

TRAIN_RAW_PATH = PROCESSED_PATH / '01_train_model_raw.csv'
VAL_RAW_PATH = PROCESSED_PATH / '01_val_model_raw.csv'

print('PROCESSED_PATH existe:', PROCESSED_PATH.exists())
print('Train raw existe:', TRAIN_RAW_PATH.exists())
print('Val raw existe:', VAL_RAW_PATH.exists())


PROCESSED_PATH existe: True
Train raw existe: True
Val raw existe: True


In [4]:
train_raw = pd.read_csv(TRAIN_RAW_PATH)
val_raw = pd.read_csv(VAL_RAW_PATH)

print('Train:', train_raw.shape)
print('Validation:', val_raw.shape)

display(train_raw.head())


Train: (176, 10)
Validation: (44, 10)


,pair_id,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label
0,train_pair_000127,java,train,071.java,021.java,data\train\java\071.java,data\train\java\021.java,\n\nimport java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swin...,\n\n\n\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*...,0
1,train_pair_000028,java,train,051.java,258.java,data\train\java\051.java,data\train\java\258.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,\n\nimport java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\n \npublic clas...,1
2,train_pair_000134,java,train,051.java,257.java,data\train\java\051.java,data\train\java\257.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,\n\nimport java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\n \npublic clas...,1
3,train_pair_000037,c,train,014.c,032.c,data\train\c\014.c,data\train\c\032.c,#include <stdio.h>\n#include <stdlib.h>\n#include <sys/time.h>\n#define MINCHAR 65\n#define MAXCHAR 122\n\n\n\nint ...,"\n\n#include <stdio.h>\n#include <stdlib.h>\n#include <strings.h>\n#include <ctype.h>\n\nint (){\n\tsystem(""wget -p ...",0
4,train_pair_000015,java,train,160.java,009.java,data\train\java\160.java,data\train\java\009.java,import java.io.*;\n\n\npublic class WatchDog\n{\npublic static void main (String[] args)\n{ String isdiff = ne...,\n\nimport java.util.*;\nimport java.text.*;\nimport java.io.*;\nimport java.*;\nimport java.net.*;\n\npublic class ...,0


In [5]:
required_columns = [
    'pair_id',
    'language',
    'file_1',
    'file_2',
    'code_1_raw',
    'code_2_raw',
    'label'
]

for df_name, df in [('train_raw', train_raw), ('val_raw', val_raw)]:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f'{df_name} no tiene estas columnas: {missing}')

print('Columnas validadas correctamente.')


Columnas validadas correctamente.


In [6]:
print('Distribución train:')
display(train_raw['label'].value_counts())

print('\nDistribución validation:')
display(val_raw['label'].value_counts())

print('\nLenguajes train:')
display(train_raw['language'].value_counts())

print('\nLenguajes validation:')
display(val_raw['language'].value_counts())


Distribución train:


label
0    88
1    88
Name: count, dtype: int64


Distribución validation:


label
1    22
0    22
Name: count, dtype: int64


Lenguajes train:


language
java    109
c        67
Name: count, dtype: int64


Lenguajes validation:


language
java    26
c       18
Name: count, dtype: int64

In [7]:
def add_raw_length_features(df):
    df = df.copy()
    df['code_1_raw_chars'] = df['code_1_raw'].fillna('').apply(len)
    df['code_2_raw_chars'] = df['code_2_raw'].fillna('').apply(len)
    df['code_1_raw_lines'] = df['code_1_raw'].fillna('').apply(lambda x: len(x.splitlines()))
    df['code_2_raw_lines'] = df['code_2_raw'].fillna('').apply(lambda x: len(x.splitlines()))
    return df

train_raw = add_raw_length_features(train_raw)
val_raw = add_raw_length_features(val_raw)

display(train_raw[['code_1_raw_chars', 'code_2_raw_chars', 'code_1_raw_lines', 'code_2_raw_lines']].describe())


,code_1_raw_chars,code_2_raw_chars,code_1_raw_lines,code_2_raw_lines
count,176.000000,176.000000,176.000000,176.000000
mean,2849.863636,2837.948864,116.977273,120.085227
std,2716.120955,2966.509713,93.440031,109.833841
min,212.000000,212.000000,12.000000,16.000000
25%,1216.750000,1222.000000,56.750000,58.000000
50%,2288.000000,2065.500000,96.000000,83.000000
75%,3528.750000,3311.000000,136.000000,137.000000
max,25470.000000,26088.000000,740.000000,776.000000


In [8]:
def light_clean_code(code):
    """
    Limpieza ligera del código.
    Conserva la estructura y los tokens importantes.
    """
    if not isinstance(code, str):
        return ''

    # Normalizar saltos de línea
    code = code.replace('\r\n', '\n').replace('\r', '\n')

    # Quitar espacios al final de cada línea
    code = '\n'.join(line.rstrip() for line in code.split('\n'))

    # Reducir líneas vacías excesivas
    code = re.sub(r'\n{3,}', '\n\n', code)

    # Quitar espacios extremos
    return code.strip()


In [9]:
def remove_c_java_comments(code):
    """
    Elimina comentarios // y /* */ sin intentar parsear el lenguaje completo.
    Es una aproximación razonable para preprocesamiento inicial.
    """
    if not isinstance(code, str):
        return ''

    # Eliminar comentarios multilínea /* ... */
    code = re.sub(r'/\*.*?\*/', ' ', code, flags=re.DOTALL)

    # Eliminar comentarios de una línea //...
    code = re.sub(r'//.*', ' ', code)

    return light_clean_code(code)


In [10]:
JAVA_C_KEYWORDS = {
    # C / C++ / Java common
    'auto', 'break', 'case', 'char', 'const', 'continue', 'default', 'do',
    'double', 'else', 'enum', 'extern', 'float', 'for', 'goto', 'if',
    'int', 'long', 'register', 'return', 'short', 'signed', 'sizeof',
    'static', 'struct', 'switch', 'typedef', 'union', 'unsigned', 'void',
    'volatile', 'while',
    # C++
    'class', 'public', 'private', 'protected', 'template', 'typename',
    'using', 'namespace', 'new', 'delete', 'this', 'virtual', 'operator',
    'include', 'iostream', 'std', 'cin', 'cout', 'endl',
    # Java
    'abstract', 'assert', 'boolean', 'byte', 'catch', 'extends', 'final',
    'finally', 'implements', 'import', 'instanceof', 'interface', 'native',
    'null', 'package', 'strictfp', 'super', 'synchronized', 'throws',
    'throw', 'transient', 'try', 'true', 'false', 'String', 'System', 'Scanner',
    'ArrayList', 'HashMap', 'Math', 'Integer', 'Double', 'Long'
}

TOKEN_PATTERN = re.compile(
    r'"(?:\\.|[^"\\])*"'              # strings dobles
    r"|'(?:\\.|[^'\\])*'"             # chars o strings simples
    r'|\b\d+(?:\.\d+)?\b'              # números enteros/decimales
    r'|\b[A-Za-z_][A-Za-z0-9_]*\b'       # identificadores / keywords
    r'|==|!=|<=|>=|\+\+|--|&&|\|\||<<|>>' # operadores compuestos
    r'|[{}\[\]();,.:?]'                 # puntuación
    r'|[+\-*/%=&|!<>^~]'                # operadores simples
)

def lexical_tokens(code):
    """
    Devuelve tokens léxicos simples a partir del código.
    """
    if not isinstance(code, str):
        return []
    return TOKEN_PATTERN.findall(code)


In [11]:
def normalize_token(token):
    if re.fullmatch(r'"(?:\\.|[^"\\])*"', token):
        return 'STR'

    if re.fullmatch(r"'(?:\\.|[^'\\])*'", token):
        return 'CHAR'

    if re.fullmatch(r'\d+(?:\.\d+)?', token):
        return 'NUM'

    if re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', token):
        if token in JAVA_C_KEYWORDS:
            return token.upper()
        return 'ID'

    return token


def normalized_token_sequence(code):
    tokens = lexical_tokens(code)
    normalized = [normalize_token(tok) for tok in tokens]
    return ' '.join(normalized)


def raw_token_sequence(code):
    tokens = lexical_tokens(code)
    return ' '.join(tokens)


In [12]:
def preprocess_pairs(df):
    df = df.copy()

    # Limpieza ligera
    df['code_1_clean'] = df['code_1_raw'].apply(light_clean_code)
    df['code_2_clean'] = df['code_2_raw'].apply(light_clean_code)

    # Sin comentarios
    df['code_1_no_comments'] = df['code_1_clean'].apply(remove_c_java_comments)
    df['code_2_no_comments'] = df['code_2_clean'].apply(remove_c_java_comments)

    # Tokens sin normalizar
    df['code_1_tokens'] = df['code_1_no_comments'].apply(raw_token_sequence)
    df['code_2_tokens'] = df['code_2_no_comments'].apply(raw_token_sequence)

    # Tokens normalizados
    df['code_1_norm'] = df['code_1_no_comments'].apply(normalized_token_sequence)
    df['code_2_norm'] = df['code_2_no_comments'].apply(normalized_token_sequence)

    # Longitudes útiles para features posteriores
    df['code_1_clean_chars'] = df['code_1_clean'].apply(len)
    df['code_2_clean_chars'] = df['code_2_clean'].apply(len)
    df['code_1_token_count'] = df['code_1_norm'].apply(lambda x: len(x.split()))
    df['code_2_token_count'] = df['code_2_norm'].apply(lambda x: len(x.split()))

    return df


train_preprocessed = preprocess_pairs(train_raw)
val_preprocessed = preprocess_pairs(val_raw)

print('Train preprocessed:', train_preprocessed.shape)
print('Validation preprocessed:', val_preprocessed.shape)

display(train_preprocessed.head(2))


Train preprocessed: (176, 26)
Validation preprocessed: (44, 26)


,pair_id,language,scenario,file_1,file_2,path_1,path_2,code_1_raw,code_2_raw,label,...,code_1_no_comments,code_2_no_comments,code_1_tokens,code_2_tokens,code_1_norm,code_2_norm,code_1_clean_chars,code_2_clean_chars,code_1_token_count,code_2_token_count
0,train_pair_000127,java,train,071.java,021.java,data\train\java\071.java,data\train\java\021.java,\n\nimport java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swin...,\n\n\n\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*...,0,...,import java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swing.te...,import java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*;\n\npub...,import java . io . * ; import java . net . * ; import java . util . * ; import javax . swing . text . html . * ; imp...,import java . util . * ; import java . net . * ; import java . io . * ; import misc . BASE64Encoder ; import javax ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . ID . ID . * ; IMPORT ID . ID . ID . ...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ; IMPORT ID . ID . * ; PUBLIC CLASS ID...,6541,2820,992,407
1,train_pair_000028,java,train,051.java,258.java,data\train\java\051.java,data\train\java\258.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,\n\nimport java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\n \npublic clas...,1,...,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,import java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\npublic class Dicti...,import java . io . * ; import java . net . * ; import java . * ; import java . Runtime . * ; import java . Object . ...,import java . awt . * ; import java . util . * ; import java . net . * ; import java . io . * ; import java . * ; pu...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; PUBLIC CLASS ID ...,5322,4292,1031,672


In [13]:
idx = 0

print('=== Código original ===')
print(train_preprocessed.loc[idx, 'code_1_raw'][:1000])

print('\n=== Código sin comentarios ===')
print(train_preprocessed.loc[idx, 'code_1_no_comments'][:1000])

print('\n=== Tokens normalizados ===')
print(train_preprocessed.loc[idx, 'code_1_norm'][:1000])


=== Código original ===


import java.io.*;
import java.net.*;
import java.util.*;

import javax.swing.text.html.*;
import javax.swing.text.html.parser.*;
import javax.swing.text.*;



public class WatchDog {
  
  private URL url = null;
  
  private int time = 0;
  
  private ArrayList images = new ArrayList();
  

  public WatchDog(String urlString, int time) {
    this.time = time;
    try{
      this.url = new URL( urlString );
    }catch(MalformedURLException mefu){
      System.out.println(mefu.toString());
    }

  }

  
  private URLConnection getConnection( ) throws IOException    {
      URLConnection  = url.openConnection();
      return  ;
  }
  
   public ArrayList parse()throws Exception{
      HTMLEditorKit.ParserCallback callback = new HTMLEditorKit.ParserCallback (){

        public void handleSimpleTag(HTML.Tag t, MutableAttributeSet a, int pos){
          if(HTML.Tag.IMG == t){
            String image = (String)a.getAttribute(HTML.Attribute.SRC);
            image =

In [14]:
def validate_preprocessed(df, df_name):
    columns_to_check = [
        'code_1_clean',
        'code_2_clean',
        'code_1_no_comments',
        'code_2_no_comments',
        'code_1_norm',
        'code_2_norm'
    ]

    for col in columns_to_check:
        missing = df[col].isna().sum()
        empty = (df[col].astype(str).str.strip() == '').sum()
        print(f'{df_name} | {col}: missing={missing}, empty={empty}')

validate_preprocessed(train_preprocessed, 'train')
print()
validate_preprocessed(val_preprocessed, 'validation')


train | code_1_clean: missing=0, empty=0
train | code_2_clean: missing=0, empty=0
train | code_1_no_comments: missing=0, empty=0
train | code_2_no_comments: missing=0, empty=0
train | code_1_norm: missing=0, empty=0
train | code_2_norm: missing=0, empty=0

validation | code_1_clean: missing=0, empty=0
validation | code_2_clean: missing=0, empty=0
validation | code_1_no_comments: missing=0, empty=0
validation | code_2_no_comments: missing=0, empty=0
validation | code_1_norm: missing=0, empty=0
validation | code_2_norm: missing=0, empty=0


In [16]:
def jaccard_similarity(text_a, text_b):
    set_a = set(str(text_a).split())
    set_b = set(str(text_b).split())

    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0

    return len(set_a & set_b) / len(set_a | set_b)


def token_overlap_ratio(text_a, text_b):
    tokens_a = str(text_a).split()
    tokens_b = str(text_b).split()

    if not tokens_a or not tokens_b:
        return 0.0

    counter_a = Counter(tokens_a)
    counter_b = Counter(tokens_b)

    overlap = sum((counter_a & counter_b).values())
    min_len = min(len(tokens_a), len(tokens_b))

    return overlap / min_len if min_len > 0 else 0.0


def add_similarity_features(df):
    df = df.copy()

    df['jaccard_norm'] = df.apply(
        lambda row: jaccard_similarity(row['code_1_norm'], row['code_2_norm']),
        axis=1
    )

    df['token_overlap_norm'] = df.apply(
        lambda row: token_overlap_ratio(row['code_1_norm'], row['code_2_norm']),
        axis=1
    )

    df['abs_token_count_diff'] = (
        df['code_1_token_count'] - df['code_2_token_count']
    ).abs()

    df['relative_token_count_diff'] = df['abs_token_count_diff'] / (
        df[['code_1_token_count', 'code_2_token_count']].max(axis=1).replace(0, 1)
    )

    return df


train_preprocessed = add_similarity_features(train_preprocessed)
val_preprocessed = add_similarity_features(val_preprocessed)

display(train_preprocessed[['label', 'jaccard_norm', 'token_overlap_norm', 'relative_token_count_diff']].head())


,label,jaccard_norm,token_overlap_norm,relative_token_count_diff
0,0,0.673077,0.970516,0.589718
1,1,0.826923,0.986607,0.348206
2,1,0.826923,0.856070,0.225024
3,0,0.486486,0.845455,0.836066
4,0,0.709677,0.821429,0.325301


In [17]:
display(
    train_preprocessed.groupby('label')[
        ['jaccard_norm', 'token_overlap_norm', 'relative_token_count_diff']
    ].mean()
)

,jaccard_norm,token_overlap_norm,relative_token_count_diff
label,,,
0,0.613969,0.864588,0.513219
1,0.871422,0.957173,0.208579


In [18]:
# 1. Dataset completo
train_preprocessed.to_csv(PROCESSED_PATH / '02_train_preprocessed.csv', index=False)
val_preprocessed.to_csv(PROCESSED_PATH / '02_val_preprocessed.csv', index=False)

# 2. Dataset para baseline clásico
baseline_columns = [
    'pair_id', 'language', 'file_1', 'file_2',
    'code_1_norm', 'code_2_norm',
    'jaccard_norm', 'token_overlap_norm', 'relative_token_count_diff',
    'label'
]

train_baseline = train_preprocessed[baseline_columns].copy()
val_baseline = val_preprocessed[baseline_columns].copy()

train_baseline.to_csv(PROCESSED_PATH / '02_train_baseline.csv', index=False)
val_baseline.to_csv(PROCESSED_PATH / '02_val_baseline.csv', index=False)

# 3. Dataset para transformer
transformer_columns = [
    'pair_id', 'language', 'file_1', 'file_2',
    'code_1_clean', 'code_2_clean',
    'label'
]

train_transformer = train_preprocessed[transformer_columns].copy()
val_transformer = val_preprocessed[transformer_columns].copy()

train_transformer.to_csv(PROCESSED_PATH / '02_train_transformer.csv', index=False)
val_transformer.to_csv(PROCESSED_PATH / '02_val_transformer.csv', index=False)

print('Archivos guardados en:', PROCESSED_PATH)


Archivos guardados en: data\processed


In [19]:
files_to_check = [
    '02_train_preprocessed.csv',
    '02_val_preprocessed.csv',
    '02_train_baseline.csv',
    '02_val_baseline.csv',
    '02_train_transformer.csv',
    '02_val_transformer.csv'
]

for file_name in files_to_check:
    path = PROCESSED_PATH / file_name
    print(file_name, '->', path.exists(), '| size:', path.stat().st_size if path.exists() else 0)


02_train_preprocessed.csv -> True | size: 4321102
02_val_preprocessed.csv -> True | size: 1124389
02_train_baseline.csv -> True | size: 435889
02_val_baseline.csv -> True | size: 101801
02_train_transformer.csv -> True | size: 989086
02_val_transformer.csv -> True | size: 258477


In [20]:
check_train_baseline = pd.read_csv(PROCESSED_PATH / '02_train_baseline.csv')
check_train_transformer = pd.read_csv(PROCESSED_PATH / '02_train_transformer.csv')

print('Baseline:', check_train_baseline.shape)
print('Transformer:', check_train_transformer.shape)

display(check_train_baseline.head())
display(check_train_transformer.head())


Baseline: (176, 10)
Transformer: (176, 7)


,pair_id,language,file_1,file_2,code_1_norm,code_2_norm,jaccard_norm,token_overlap_norm,relative_token_count_diff,label
0,train_pair_000127,java,071.java,021.java,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . ID . ID . * ; IMPORT ID . ID . ID . ...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ; IMPORT ID . ID . * ; PUBLIC CLASS ID...,0.673077,0.970516,0.589718,0
1,train_pair_000028,java,051.java,258.java,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; PUBLIC CLASS ID ...,0.826923,0.986607,0.348206,1
2,train_pair_000134,java,051.java,257.java,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID ....,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; PUBLIC CLASS ID ...,0.826923,0.856070,0.225024,1
3,train_pair_000037,c,014.c,032.c,"INCLUDE < ID . ID > INCLUDE < ID . ID > INCLUDE < ID / ID . ID > ID ID NUM ID ID NUM INT ID ( INT ID , INT * ID ) ; ...",INCLUDE < ID . ID > INCLUDE < ID . ID > INCLUDE < ID . ID > INCLUDE < ID . ID > INT ( ) { ID ( STR ID ID STR ID ID ....,0.486486,0.845455,0.836066,0
4,train_pair_000015,java,160.java,009.java,IMPORT ID . ID . * ; PUBLIC CLASS ID { PUBLIC STATIC VOID ID ( STRING [ ] ID ) { STRING ID = NEW STRING ( ) ; STRING...,IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . ID . * ; IMPORT ID . * ; IMPORT ID . ID . * ; PUBLIC CLASS ID ...,0.709677,0.821429,0.325301,0


,pair_id,language,file_1,file_2,code_1_clean,code_2_clean,label
0,train_pair_000127,java,071.java,021.java,import java.io.*;\nimport java.net.*;\nimport java.util.*;\n\nimport javax.swing.text.html.*;\nimport javax.swing.te...,import java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport misc.BASE64Encoder;\nimport javax.swing.*;\n\npub...,0
1,train_pair_000028,java,051.java,258.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,import java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\npublic class Dicti...,1
2,train_pair_000134,java,051.java,257.java,import java.io.*;\nimport java.net.*;\nimport java.*;\nimport java.Runtime.*;\nimport java.Object.*;\nimport java.ut...,import java.awt.*;\nimport java.util.*;\nimport java.net.*;\nimport java.io.*;\nimport java.*;\n\npublic class Brute...,1
3,train_pair_000037,c,014.c,032.c,#include <stdio.h>\n#include <stdlib.h>\n#include <sys/time.h>\n#define MINCHAR 65\n#define MAXCHAR 122\n\nint brut...,"#include <stdio.h>\n#include <stdlib.h>\n#include <strings.h>\n#include <ctype.h>\n\nint (){\n\tsystem(""wget -p --co...",0
4,train_pair_000015,java,160.java,009.java,import java.io.*;\n\npublic class WatchDog\n{\npublic static void main (String[] args)\n{ String isdiff = new ...,import java.util.*;\nimport java.text.*;\nimport java.io.*;\nimport java.*;\nimport java.net.*;\n\npublic class Watc...,0
